In [ ]:
%pip install -qU 'sagemaker>=2.200.0,<3.0.0' boto3 pandas numpy

In [ ]:
import io
import json
import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker.session import Session
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from sagemaker.predictor import Predictor

In [ ]:
ACCOUNT_ID   = boto3.client("sts").get_caller_identity()["Account"]
REGION       = boto3.session.Session().region_name
BUCKET       = f"multiagentfrauddetection-{ACCOUNT_ID}"
RAW_KEY      = "fraud/data/transactions_train_ext.csv"
PREFIX       = "fraud/prepared"
ENDPOINT_NAME = "fraud-xgb-ext12"

# Derived
boto_sess = boto3.Session(region_name=REGION)
sm_sess   = sagemaker.Session(boto_session=boto_sess)
role      = sagemaker.get_execution_role()
s3        = boto3.client("s3", region_name=REGION)

print(f"Account:  {ACCOUNT_ID}")
print(f"Region:   {REGION}")
print(f"Bucket:   {BUCKET}")
print(f"Raw CSV:  s3://{BUCKET}/{RAW_KEY}")
print(f"Role:     {role}")

In [ ]:
# Read from S3 using boto3 (avoids sagemaker v3 async issues)
obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
df_raw = pd.read_csv(io.BytesIO(obj["Body"].read()))

print("CSV columns:", df_raw.shape[1])
print("First row:", df_raw.iloc[0].tolist())
print("Headers:", list(df_raw.columns))
df_raw.head(3)

# label first, NO header
cols = ["label",
        "amount","hour","dow",
        "src_cnt_1h","src_sum_1h","src_near_thr_1h","prior_pair_count",
        "dst_cnt_1h","dst_sum_1h","dst_distinct_src_1h",
        "geo_change_flag","device_change_flag"]

# If the CSV has headers, use them; if headerless, assign names
if list(df_raw.columns[:3]) == ['label', 'amount', 'hour']:
    df = df_raw[cols]
else:
    df_raw.columns = cols
    df = df_raw

feature_cols = cols[1:]
label_col = "label"

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df[label_col])

train_xgb = pd.concat([train_df[[label_col]], train_df[feature_cols]], axis=1)
val_xgb   = pd.concat([val_df[[label_col]],   val_df[feature_cols]],   axis=1)

train_xgb.to_csv("train.csv", index=False, header=False)
val_xgb.to_csv("validation.csv", index=False, header=False)

print("Train shape:", train_xgb.shape, "Val shape:", val_xgb.shape)


In [ ]:
train_s3 = sm_sess.upload_data("train.csv", bucket=BUCKET, key_prefix=PREFIX)
val_s3   = sm_sess.upload_data("validation.csv", bucket=BUCKET, key_prefix=PREFIX)

print(f"Train -> {train_s3}")
print(f"Val   -> {val_s3}")

In [ ]:
container = sagemaker.image_uris.retrieve("xgboost", REGION, "latest")

s3_train = TrainingInput(s3_data=f"s3://{BUCKET}/{PREFIX}/train.csv", content_type="text/csv")
s3_val   = TrainingInput(s3_data=f"s3://{BUCKET}/{PREFIX}/validation.csv", content_type="text/csv")

xgb = sagemaker.estimator.Estimator(
container,
role,
instance_count=1,
instance_type="ml.m4.xlarge",
output_path=f"s3://{BUCKET}/{PREFIX}/output",
sagemaker_session=sm_sess,
)

xgb.set_hyperparameters(
objective="binary:logistic",
eval_metric="auc",
num_round=400,
max_depth=6,
eta=0.2,
subsample=0.8,
colsample_bytree=0.8,
)

xgb.fit({"train": s3_train, "validation": s3_val})                                         

In [ ]:
print("model_data:", xgb.model_data)          # full s3://.../model.tar.gz
print("region:", REGION)
role = sagemaker.get_execution_role()

In [ ]:
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
)

predictor.serializer = CSVSerializer()

In [ ]:
# Connect to existing endpoint
predictor = Predictor(endpoint_name=ENDPOINT_NAME, sagemaker_session=sm_sess)
predictor.serializer = CSVSerializer()

event = [102.5, 14, 2, 9, 850.0, 5, 1, 7, 620.0, 5, 1, 0]
prob = predictor.predict(event)
print("Fraud probability:", prob)

In [ ]:
#predictor.delete_endpoint()
